In [1]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, pi

In [2]:
c    = 2.99792458e10
AU   = 1.495978707e13
Rsun = 6.957e10
Msun = 1.98847e33
G    = 6.67430e-8

VESCSURF = np.sqrt(2*G*Msun/Rsun)/c   # ~2.06e-3
VESCAU   = np.sqrt(2*G*Msun/AU)/c

#Model parameters
MCHI_MeV  = 0.5         # DM mass (MeV); pick e.g. 0.5 MeV as in their plots
ME_MeV    = 0.510998    # electron mass
SIGMA_T   = 1e-34       # cm^2 (overall scale)
N_SHELL   = 300         # thin shells inside the Sun (2000 in the paper; 300–600 is fine)
N_EVENTS  = 2000        # number of rays to shoot
B_MAX     = 4.0*Rsun    # focusing aperture
V0_HALO   = 1.0e-3      # halo dispersion (paper: ~1e-3 c)
VMAX_HALO = 2.0e-3      # halo cutoff (paper: ~2e-3 c)
SEED      = 1

In [ ]:
rng = np.random.default_rng(SEED)

# ------------------- (SWAPPABLE) solar profiles -------------------
# Replace these three with your SSM profiles to get publication-grade agreement.

def T_of_r(r):
    """plasma temperature in keV."""
    x = r/Rsun
    T0, Ts = 1.35, 0.02               # ~1.35 keV at center to ~0.02 keV at surface
    return Ts + (T0-Ts)*(1 - x**2)**2 # smooth quartic

def ne_of_r(r):
    """electron number density in cm^-3."""
    x = r/Rsun
    n0 = 6e25
    return n0*np.exp(-10*x - 2*x**2)

def vesc_of_r(r):
    """escape speed in units of c; uniform-density sphere proxy (analytic).
       v_esc^2(r) = (v_esc^2(R)/2)*(3 - (r/R)^2).  Matches v_esc(R) and gives ~1.5× at center."""
    return np.sqrt( (VESCSURF**2)/2.0 * (3.0 - (r/Rsun)**2) )

# ------------------- ANPPR formulas -------------------
def F_of_x(x):
    return np.sqrt(1.0/(2.0*pi))*( 2.0*np.exp(-0.5*x*x) + np.sqrt(2.0*pi)*(1.0+x*x)*erf(x/np.sqrt(2.0)) )

def Gamma_total(r, vchi):
    """Single-scatter rate per incoming DM, Eq. (12)."""
    T = T_of_r(r)
    return SIGMA_T * ne_of_r(r) * vchi * np.sqrt(T/ME_MeV) * F_of_x(vchi/np.sqrt(T/ME_MeV))

def kernel_q_costh(q, costh, k1, mchi, me, T):
    """Eq. (11) up to an overall constant: q * exp[-((me/2mχ)(2 k1 cosθ + q) + q/2)^2 /(2 me T)]."""
    term = (me/(2.0*mchi))*(2.0*k1*costh + q) + 0.5*q
    return q * np.exp( - (term*term)/(2.0*me*T) )

def sample_q_costh(k1, mchi, me, T, _rng):
    """Rejection sampling for (q, cosθ_q, φ) with a cheap adaptive envelope."""
    q_scale = np.sqrt(2.0*me*T) * (1.0 + me/mchi)
    q_max   = 8.0*q_scale
    # cheap envelope height
    qs = np.linspace(0, q_max, 80)
    cs = np.linspace(-1, 1, 60)
    QQ, CC = np.meshgrid(qs, cs, indexing='ij')
    Kmax = float(kernel_q_costh(QQ, CC, k1, mchi, me, T).max()) + 1e-300
    for _ in range(2000):
        q     = q_max * _rng.random()
        costh = 2.0*_rng.random() - 1.0
        if Kmax*_rng.random() <= kernel_q_costh(q, costh, k1, mchi, me, T):
            phi = 2.0*pi*_rng.random()
            return q, costh, phi
    # very rare fallback
    return q_scale, 0.0, 2.0*pi*_rng.random()

# ------------------- geometric helpers -------------------
def rotate_to_local(v, er):
    """Return v in a frame with e_r=(1,0,0) and the rotation matrix R (columns = basis)."""
    er = er/np.linalg.norm(er)
    tmp = np.array([0.0,1.0,0.0]) if abs(er[1])<0.9 else np.array([1.0,0.0,0.0])
    et1 = np.cross(er, tmp); et1 /= np.linalg.norm(et1)
    et2 = np.cross(er, et1)
    R = np.stack([er, et1, et2], axis=1)
    return R.T @ v, R

def rotate_from_local(vloc, R):
    return R @ vloc

# ------------------- halo sampling -------------------
def sample_vinf(rng):
    # truncated Maxwell in speed: f(v) ∝ v^2 exp(-v^2/(2 v0^2)), v < VMAX_HALO
    for _ in range(5000):
        v = VMAX_HALO*rng.random()
        f = (v*v) * np.exp(-v*v/(2.0*V0_HALO*V0_HALO))
        fmax = (np.sqrt(2.0)*V0_HALO)**2 * np.exp(-1.0)
        if rng.random() < f/fmax:
            return v
    return VMAX_HALO*0.7

def sample_entry(rng):
    # isotropic point on the sphere
    z   = 2*rng.random()-1.0
    phi = 2.0*pi*rng.random()
    er  = np.array([np.sqrt(1-z*z)*np.cos(phi), np.sqrt(1-z*z)*np.sin(phi), z])
    # orthonormal tangential basis
    ytmp = np.array([0,1.0,0]) if abs(er[1])<0.9 else np.array([1.0,0,0])
    et1  = np.cross(ytmp, er); et1 /= np.linalg.norm(et1)
    et2  = np.cross(er, et1)
    return er, et1, et2

# ------------------- MC core -------------------
def run_mc(n_events=N_EVENTS, seed=SEED):
    rng = np.random.default_rng(seed)
    mchi, me = MCHI_MeV, ME_MeV
    r_edges  = np.linspace(0.0, Rsun, N_SHELL+1)

    E_esc_at_AU = []

    for _ in range(n_events):
        v_inf = sample_vinf(rng)
        b     = np.sqrt(rng.random())*B_MAX
        h     = v_inf * b                                 # specific angular momentum
        v_R   = np.sqrt(v_inf*v_inf + VESCSURF*VESCSURF)  # speed at R⊙ by energy conservation
        vt_R  = h/Rsun
        if vt_R >= v_R:        # misses the Sun
            continue
        vr_R  = -np.sqrt(max(1e-30, v_R*v_R - vt_R*vt_R)) # inward

        er, et1, et2 = sample_entry(rng)
        r_vec = er*Rsun
        v_vec = vr_R*er + vt_R*et2

        # straight segments inside shells; gravity applied at boundaries
        while True:
            r    = np.linalg.norm(r_vec)
            vmag = np.linalg.norm(v_vec)
            if r<=0 or vmag<=1e-20:
                break

            vloc, R = rotate_to_local(v_vec, r_vec)
            cos_a    = vloc[0]/vmag
            idx      = int(np.clip(np.searchsorted(r_edges, r, side='right') - 1, 0, N_SHELL-1))
            r_in, r_out = r_edges[idx], r_edges[idx+1]

            def s_to(rb):
                under = rb*rb - r*r*(1.0 - cos_a*cos_a)
                if under <= 0: return np.inf
                return - r*cos_a + np.sqrt(under)

            s_in  = s_to(r_in)  if vloc[0] < 0 else np.inf
            s_out = s_to(r_out) if vloc[0] > 0 else np.inf
            s_next = min(s for s in (s_in, s_out) if np.isfinite(s))

            # scattering probability in this segment
            nhat  = v_vec/vmag
            r_mid = np.linalg.norm(r_vec + 0.5*s_next*nhat)
            Gamma = Gamma_total(r_mid, vmag)
            P     = min(0.2, Gamma * (s_next/vmag))       # thin-shell assumption

            if rng.random() < P:
                # move to mid-segment and scatter
                r_vec = r_vec + 0.5*s_next*nhat
                r     = np.linalg.norm(r_vec)
                # update speed by gravity along the drift (E = const): v^2 + v_esc^2(r) = const
                vmag  = np.sqrt(max(1e-30, vmag*vmag + vesc_of_r(r-1e-6)**2 - vesc_of_r(r)**2))
                v_vec = nhat*vmag

                # draw (q, cosθ_q, φ) from Eq. (11)
                T  = T_of_r(r)
                k1 = mchi*vmag
                q, costh, phi = sample_q_costh(k1, mchi, me, T, rng)

                k1hat = v_vec/np.linalg.norm(v_vec)
                # orthonormal basis around k1hat
                a  = np.array([1,0,0]) if abs(k1hat[0])<0.9 else np.array([0,1,0])
                e2 = np.cross(k1hat, a); e2 /= np.linalg.norm(e2)
                e3 = np.cross(k1hat, e2)
                q_vec = q*(costh*k1hat + np.sqrt(max(0.0,1.0-costh*costh))*(np.cos(phi)*e2 + np.sin(phi)*e3))
                k2_vec = k1hat*k1 - q_vec
                v_vec  = k2_vec/mchi
                continue

            # otherwise go to the next spherical boundary
            r_prev = r
            r_vec  = r_vec + s_next*nhat
            r      = np.linalg.norm(r_vec)

            # gravity at the boundary: E const between collisions => v^2 + v_esc^2(r) = const
            vmag_new = np.sqrt(max(1e-30, vmag*vmag + vesc_of_r(r_prev)**2 - vesc_of_r(r)**2))

            # conserve specific angular momentum (between collisions)
            vloc, R = rotate_to_local(v_vec, r_vec)
            vt      = np.sqrt(max(0.0, vloc[1]**2 + vloc[2]**2))
            if vt>0:
                h_spec  = r * vt
                vt_new  = min(vmag_new, h_spec/(r+1e-30))
                t_dir   = np.array([0.0, vloc[1], vloc[2]])
                t_dir  /= (np.linalg.norm(t_dir)+1e-30)
                v_rad_s = np.sign(vloc[0]) if abs(vloc[0])>1e-12 else (-1.0 if s_in < s_out else 1.0)
                v_rad_n = v_rad_s*np.sqrt(max(1e-30, vmag_new*vmag_new - vt_new*vt_new))
                vloc_new = np.array([v_rad_n, vt_new*t_dir[1], vt_new*t_dir[2]])
            else:
                vloc_new = np.array([np.sign(vloc[0])*vmag_new, 0.0, 0.0])
            v_vec = rotate_from_local(vloc_new, R)

            # leaving the Sun?
            if r >= Rsun-1e-6 and np.dot(v_vec, r_vec) > 0:
                v_out = np.linalg.norm(v_vec)
                if v_out > VESCSURF:
                    # energy at 1 AU by energy conservation
                    v_AU = np.sqrt(max(0.0, v_out*v_out - 2.0*G*Msun/c**2*(1.0/Rsun - 1.0/AU)))
                    E_AU = 0.5*MCHI_MeV*(v_AU**2)  # MeV
                    E_esc_at_AU.append(E_AU)
                break

    return np.array(E_esc_at_AU)

# ------------------- run + plots -------------------
E = run_mc()

if E.size == 0:
    print("No escaping particles recorded — increase N_EVENTS or SIGMA_T.")
else:
    nbins  = 60
    Emax   = max(1e-6, np.percentile(E, 99.5))
    edges  = np.linspace(0, Emax, nbins+1)
    hist, _ = np.histogram(E, bins=edges, density=True)   # normalized: ∫ dE hist = 1
    centers = 0.5*(edges[:-1]+edges[1:])

    # Flux at Earth (shape): dΦ_reflected/dE = Φ_halo × F_Aρ(E) × Aρ /(4π AU^2)
    A_rho = pi*(4.0*Rsun)**2
    geom  = A_rho/(4.0*pi*AU*AU)   # include geometry; Φ_halo left as separate prefactor
    dPhi_shape = hist*geom

    plt.figure(figsize=(7.0,4.6))
    plt.step(centers, hist, where='mid', lw=2, label=r'$F_{A_\rho}(E)$')
    plt.xlabel('E at 1 AU [MeV]'); plt.ylabel('Probability density')
    plt.title('Reflected DM energy distribution (contact, MC)'); plt.legend(); plt.tight_layout()

    plt.figure(figsize=(7.0,4.6))
    plt.step(centers, dPhi_shape, where='mid', lw=2)
    plt.xlabel('E at 1 AU [MeV]'); plt.ylabel(r'$d\Phi_{\rm reflected}/dE$  (arb. units)')
    plt.title('Flux at Earth (geometry factor included; $\\Phi_{\\rm halo}$ omitted)')
    plt.tight_layout()
    plt.show()